In [20]:
from __future__ import annotations

import json
import logging
from pathlib import Path
import torch

# Repo is editable-installed into the `circuitTracer` env, so these import
# from anywhere — no sys.path manipulation needed.
import macag
import macag.scoring
from circuit_tracer import ReplacementModel, attribute
from macag.graph import CircuitGraph

REPO_ROOT = Path(macag.__file__).resolve().parents[1]
print(f"macag loaded from: {macag.__file__}")
print(f"repo root: {REPO_ROOT}")

macag loaded from: /Users/ssuresh/circuit-tracer/macag/__init__.py
repo root: /Users/ssuresh/circuit-tracer


In [21]:
CLT_MODEL_NAME = "gemma2-426k"
CLT_MODEL = "google/gemma-2-2b"
CLT_TSET = "mntss/clt-gemma-2-2b-426k"
PROMPT = "Fact: The capital city of the country that has the highest life expectancy is "
TARGET = " Oslo"
FOIL = " Paris"
DTYPE = getattr(torch, "bfloat16")
BATCH_SIZE = 1
MAX_FEATURE_NODES = 50

SLUG = "gemma2-426k-acdc-game1-oslo-paris"
OUTDIR = "results/example_test/gemma2-426k/gemma2-426k-acdc-game1-oslo-paris"
DEVICE = "cpu"
NODE_THRESHOLD = 0.8
EDGE_THRESHOLD = 0.98

In [22]:
from circuit_tracer.utils.hf_utils import load_transcoder_from_hub
transcoder, config = load_transcoder_from_hub(CLT_TSET, dtype=DTYPE, lazy_encoder=False, lazy_decoder=True)


Fetching 52 files:   0%|          | 0/52 [00:00<?, ?it/s]

In [23]:
from circuit_tracer import ReplacementModel, attribute

model = ReplacementModel.from_pretrained_and_transcoders(CLT_MODEL, transcoder, dtype=DTYPE, backend="transformerlens")
print(model)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
TransformerLensReplacementModel(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-25): 26 x TransformerBlock(
      (ln1): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln1_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2_post): RMSNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): ReplacementMLP(
 

In [24]:
from circuit_tracer import attribute

graph = attribute(prompt=PROMPT, model=model, batch_size=BATCH_SIZE, max_feature_nodes=MAX_FEATURE_NODES, verbose=True)
print(graph)

Phase 0: Precomputing activations and vectors
Precomputation completed in 30.42s
Found 4658 active features
Phase 1: Running forward pass
Forward pass completed in 19.57s
Phase 2: Building input vectors
Selected 10 logits with cumulative probability 0.9102
Will include 50 of 4658 feature nodes
Input vectors built in 0.09s
Phase 3: Computing logit attributions
Logit attributions completed in 7.69s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 50/50 [00:17<00:00,  2.81it/s]
Feature attributions completed in 17.78s
Attribution completed in 75.55s


In [25]:
from circuit_tracer.utils.create_graph_files import create_graph_files
graph_dir = Path(OUTDIR) / "graphs"
create_graph_files(graph_or_path=graph, slug=SLUG, scan=None, output_path=str(graph_dir), node_threshold=NODE_THRESHOLD, edge_threshold=EDGE_THRESHOLD,)

In [26]:
graph_json = graph_dir / f"{SLUG}.json"
print(f"Wrote graph to {graph_json}")

Wrote graph to results/example_test/gemma2-426k/gemma2-426k-acdc-game1-oslo-paris/graphs/gemma2-426k-acdc-game1-oslo-paris.json


In [27]:
import json

graph = json.load(open(graph_json))
print(graph['nodes'])
print(graph['links'])



[{'node_id': '0_3314_15', 'feature': 5496269, 'layer': '0', 'ctx_idx': 15, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '0_3314-0', 'clerp': '', 'influence': 0.7957623600959778, 'activation': 3.03125}, {'node_id': '0_4007_15', 'feature': 8034035, 'layer': '0', 'ctx_idx': 15, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '0_4007-0', 'clerp': '', 'influence': 0.6600422859191895, 'activation': 2.75}, {'node_id': '0_7081_15', 'feature': 25080902, 'layer': '0', 'ctx_idx': 15, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '0_7081-0', 'clerp': '', 'influence': 0.4520151913166046, 'activation': 14.9375}, {'node_id': '0_10184_15', 'feature': 51872204, 'layer': '0', 'ctx_idx': 15, 'feature_type': 'cross layer transcoder', 'token_pro

In [28]:
# from IPython.display import IFrame, display
# from circuit_tracer.frontend.local_server import serve

# PORT = 8041
# graph_dir = Path(OUTDIR) / "graphs"  # must contain graph-metadata.json + {SLUG}.json

# server = serve(data_dir=str(graph_dir.resolve()), port=PORT)

# url = f"http://localhost:{PORT}/index.html?slug={SLUG}"
# print(f"Open in browser: {url}")
# display(IFrame(src=url, width="100%", height="800px"))

In [29]:
oracle_kwargs = {
    "model_name": CLT_MODEL,                    # e.g. "google/gemma-2-2b"
    "transcoder_set": CLT_TSET,     # e.g. "mntss/clt-gemma-2-2b-426k"
    "prompt": PROMPT,
    "graph_json": "/Users/ssuresh/circuit-tracer/macag/examples/results/example_test/gemma2-426k/gemma2-426k-acdc-game1-oslo-paris/graphs/gemma2-426k-acdc-game1-oslo-paris.json",                    # path to attribution graph JSON
    "backend": "transformerlens",
    "score_kind": "logit_gap",
    "strict_single_token": False,
    "freeze_attention": True,
    "target_token_by_label": {
        "y": TARGET,       # e.g. " Austin"  — note leading space for BPE
        "y_foil": FOIL,    # e.g. " Texas"
    },
    "foil_by_target": {"y": "y_foil", "y_foil": "y"},
    "model_kwargs": {"dtype": DTYPE, "device": DEVICE},
}

In [30]:
from macag.factories.replacement_model import create_replacement_model_oracle

graph_json = graph_dir / f"{SLUG}.json"
graph = CircuitGraph.from_json(graph_json)

factory_out = create_replacement_model_oracle(**oracle_kwargs)
oracle = factory_out.oracle
candidates = factory_out.candidates
print(f"Graph nodes: {len(graph.nodes())}, intervention candidates: {len(candidates)}")


Fetching 52 files:   0%|          | 0/52 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer
Graph nodes: 81, intervention candidates: 39


## Game 1 walkthrough: one greedy iteration

Game 1 finds a small evidence set `E` that keeps the target behavior faithful under circuit interventions.

Each iteration:
1. Score the baseline interventions (`S_all`, `S_empty`).
2. Compute faithfulness metrics for the current set `E`.
3. Try adding each remaining candidate and pick the one with highest **utility gain**.
4. Stop when budget / faithfulness threshold / no improving candidate.

Below we run **iteration 0** manually (`E = ∅`), then call `solve_game1` for the full hill-climb.


In [31]:
# Game 1 hyperparameters (shared with the full solve below)
GAME1_ALPHA = 0.01
GAME1_LAM = 0.01
GAME1_BUDGET = 100
GAME1_FAITHFULNESS_EPS = 0.01
GAME1_STOP_METRIC = "raw_relative"
GAME1_PREFILTER_TOP_K = 5
GAME1_CONNECTED = True
GAME1_MIN_GAIN = 0.01
TARGET_ID = "y"

oracle.reset_stats()


### Step 1: Oracle scoring primitives

For target `y`, the oracle exposes four intervention scores:
- `S_all(y)` — full circuit active
- `S_empty(y)` — all feature nodes ablated (error nodes remain)
- `S_keep(E, y)` — only subset `E` kept active
- `S_remove(E, y)` — subset `E` ablated from the full circuit


In [32]:
s_all = oracle.all(TARGET_ID)
s_empty = oracle.empty(TARGET_ID)
print(f"S_all  ({TARGET_ID}) = {s_all:.6f}")
print(f"S_empty({TARGET_ID}) = {s_empty:.6f}")
print(f"Recoverable range (S_all - S_empty) = {s_all - s_empty:.6f}")


S_all  (y) = 0.656250
S_empty(y) = -5.968750
Recoverable range (S_all - S_empty) = 6.625000


### Step 2: Faithfulness metrics for the current evidence set

For set `A`:
- **sufficiency** = `S_keep(A) - S_empty` — how much signal remains with only `A`
- **necessity** = `S_all - S_remove(A)` — how much `A` matters when removed
- **faithfulness_delta** = `α·sufficiency + (1-α)·necessity`
- **utility** = `faithfulness_delta - λ·|A|`

Start with `E = ∅`.


In [33]:
from macag.utils.metrics import compute_faithfulness_metrics, game1_utility

selected: set = set()  # current evidence E

current_metrics = compute_faithfulness_metrics(
    oracle=oracle, target=TARGET_ID, nodes=selected, alpha=GAME1_ALPHA
)
current_utility = game1_utility(
    faithfulness_delta=current_metrics.faithfulness_delta,
    size=len(selected),
    lam=GAME1_LAM,
)

print("E =", selected or "∅")
print(f"  keep_only = {current_metrics.keep_only_score:.6f}")
print(f"  remove    = {current_metrics.remove_score:.6f}")
print(f"  sufficiency        = {current_metrics.sufficiency:.6f}")
print(f"  necessity          = {current_metrics.necessity:.6f}")
print(f"  faithfulness_delta = {current_metrics.faithfulness_delta:.6f}")
print(f"  utility            = {current_utility:.6f}")


E = ∅
  keep_only = -5.968750
  remove    = 0.656250
  sufficiency        = 0.000000
  necessity          = 0.000000
  faithfulness_delta = 0.000000
  utility            = 0.000000


In [40]:
print(f"Candidates: {candidates}")

Candidates: ['0_10184_15', '0_15844_16', '0_16252_16', '0_1663_16', '0_3096_16', '0_3314_15', '0_4007_15', '0_4135_16', '0_7081_15', '0_7081_16', '0_8369_16', '0_8807_16', '10_6146_15', '12_451_16', '13_5422_16', '19_10597_16', '19_2360_16', '19_954_16', '1_3148_15', '1_3148_16', '1_3872_14', '1_3872_15', '1_3872_16', '1_5013_16', '21_3523_16', '22_10872_16', '22_9274_16', '22_9511_16', '23_11508_16', '23_13364_16', '23_25_16', '23_9158_16', '23_9826_16', '24_11985_16', '24_13002_16', '24_4225_16', '24_8525_16', '25_13582_16', '3_3783_16']


### Step 3: Prefilter candidates (optional)

With many graph nodes, rank singleton utility gains and keep top-k before the greedy sweep. This matches `prefilter_top_k` in the solver.


In [34]:
from macag.games.game1_min_faithful import prefilter_candidates

candidate_pool = prefilter_candidates(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    candidates=candidates,
    alpha=GAME1_ALPHA,
    lam=GAME1_LAM,
    top_k=GAME1_PREFILTER_TOP_K,
    connected=GAME1_CONNECTED,
    progress=True,
)
print(f"Prefiltered {len(candidates)} -> {len(candidate_pool)} candidates")
for i, node in enumerate(candidate_pool[:5]):
    print(f"  [{i}] {node}")
if len(candidate_pool) > 5:
    print(f"  ... ({len(candidate_pool) - 5} more)")


Game1 prefilter: 100%|██████████| 39/39 [1:16:17<00:00, 117.38s/it]

Prefiltered 39 -> 5 candidates
  [0] 0_8369_16
  [1] 24_13002_16
  [2] 0_7081_16
  [3] 1_3148_16
  [4] 1_3872_16


### Step 4: Greedy sweep — pick the best marginal gain

For each candidate `n ∉ E`, form `E ∪ {n}`, recompute utility, and keep the node with largest **gain** = `U(E∪{n}) - U(E)`.

When `connected=True`, skip trials that would make `E` disconnected.


In [38]:
from macag.graph import NodeId


def _sort_key(node: NodeId) -> str:
    return str(node)


best_node = None
best_gain = GAME1_MIN_GAIN
best_faith_gain = 0.0
sweep_rows = []

for node in candidate_pool:
    if node in selected:
        continue
    trial = set(selected)
    trial.add(node)
    if GAME1_CONNECTED and len(trial) > 1 and not graph.connected_through(trial):
        continue
    trial_metrics = compute_faithfulness_metrics(
        oracle=oracle, target=TARGET_ID, nodes=trial, alpha=GAME1_ALPHA
    )
    trial_utility = game1_utility(
        faithfulness_delta=trial_metrics.faithfulness_delta,
        size=len(trial),
        lam=GAME1_LAM,
    )
    gain = trial_utility - current_utility
    faith_gain = trial_metrics.faithfulness_delta - current_metrics.faithfulness_delta
    sweep_rows.append((gain, faith_gain, node))
    if gain > best_gain:
        best_gain = gain
        best_node = node
        best_faith_gain = faith_gain
    elif gain == best_gain and best_node is not None and _sort_key(node) < _sort_key(best_node):
        best_node = node
        best_faith_gain = faith_gain

sweep_rows.sort(key=lambda r: (-r[0], _sort_key(r[2])))
print(f"Sweep evaluated {len(sweep_rows)} connected candidates")
print(f"Current utility: {current_utility:.6f}")
print()
for gain, faith_gain, node in sweep_rows[:5]:
    marker = " <-- winner" if node == best_node else ""
    print(f"  {node}: utility_gain={gain:.6f}, faith_gain={faith_gain:.6f}{marker}")

if best_node is None:
    print("\nNo improving candidate — solver would stop here.")
else:
    print(f"\nBest node: {best_node}  (utility gain = {best_gain:.6f}, faith gain = {best_faith_gain:.6f})")


Sweep evaluated 4 connected candidates
Current utility: 0.000000

  24_13002_16: utility_gain=5.359688, faith_gain=5.379688 <-- winner
  1_3872_16: utility_gain=4.797500, faith_gain=4.817500
  0_7081_16: utility_gain=3.102500, faith_gain=3.122500
  1_3148_16: utility_gain=3.076875, faith_gain=3.096875

Best node: 24_13002_16  (utility gain = 5.359688, faith gain = 5.379688)


### Step 5: Add the winner (iteration 0 → 1)

The solver adds `best_node` to `E` and records `first_faith_gain` for the `raw_relative` stop rule.

Stop checks (usually not triggered on the first add):
- **raw_relative**: stop if next best `faith_gain < eps × first_faith_gain`
- **normalized**: stop if `faithfulness_delta_normalized ≥ 1 - eps`


In [39]:
if best_node is not None:
    selected.add(best_node)
    selected_order = [best_node]
    first_faith_gain = best_faith_gain

    new_metrics = compute_faithfulness_metrics(
        oracle=oracle, target=TARGET_ID, nodes=selected, alpha=GAME1_ALPHA
    )
    new_utility = game1_utility(
        faithfulness_delta=new_metrics.faithfulness_delta,
        size=len(selected),
        lam=GAME1_LAM,
    )

    print(f"After adding {best_node}:")
    print(f"  E = {selected}")
    print(f"  faithfulness_delta            = {new_metrics.faithfulness_delta:.6f}")
    print(f"  faithfulness_delta_normalized = {new_metrics.faithfulness_delta_normalized:.6f}")
    print(f"  utility                       = {new_utility:.6f}")
    print(f"  first_faith_gain (for stop)   = {first_faith_gain:.6f}")

    stats = oracle.cache_stats()
    print(f"\nOracle calls so far: {stats['oracle_calls']}, cache hits: {stats['cache_hits']}")
else:
    print("Skipped — no improving candidate in step 4.")


After adding 24_13002_16:
  E = {'24_13002_16', '0_8369_16'}
  faithfulness_delta            = 5.379688
  faithfulness_delta_normalized = 0.812028
  utility                       = 5.359688
  first_faith_gain (for stop)   = 5.379688

Oracle calls so far: 8, cache hits: 208


## Run full Game 1

Now run the complete greedy hill-climb (all iterations, caching, and stopping rules).


In [41]:
from macag.games.game1_min_faithful import solve_game1

result_game1 = solve_game1(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    candidates=candidates,
    alpha=GAME1_ALPHA,
    lam=GAME1_LAM,
    budget=GAME1_BUDGET,
    faithfulness_eps=GAME1_FAITHFULNESS_EPS,
    stop_metric=GAME1_STOP_METRIC,
    prefilter_top_k=GAME1_PREFILTER_TOP_K,
    connected=GAME1_CONNECTED,
    min_gain=GAME1_MIN_GAIN,
    progress=True,
    log_every=10,
)

print(f"Evidence size: {len(result_game1.evidence)}")
print(f"Selected order (first 10): {result_game1.selected_order[:10]}")
print(f"Final utility: {result_game1.utility:.6f}")
print(f"Faithfulness delta: {result_game1.metrics.faithfulness_delta:.6f}")
print(f"Oracle calls: {result_game1.oracle_calls}, cache hits: {result_game1.cache_hits}")


Game1 prefilter: 100%|██████████| 39/39 [00:00<00:00, 68072.35it/s]
                                                        

Evidence size: 1
Selected order (first 10): ['0_8369_16']
Final utility: 5.362813
Faithfulness delta: 5.372813
Oracle calls: 0, cache hits: 196


# Visualize the nodes identified by MACAG game 1

In [ ]:
# from IPython.display import IFrame, display
# from circuit_tracer.frontend.local_server import serve

# PORT = 8041
# graph_dir = Path(OUTDIR) / "graphs"  # must contain graph-metadata.json + {SLUG}.json

# server = serve(data_dir=str(graph_dir.resolve()), port=PORT)

# url = f"http://localhost:{PORT}/index.html?slug={SLUG}"
# print(f"Open in browser: {url}")
# display(IFrame(src=url, width="100%", height="800px"))

## Game 2 walkthrough: contrastive evidence allocation

Game 2 finds **two** evidence sets — one for the target `y`, one for the foil `y_foil` — that are faithful to their respective behaviors while staying **non-overlapping**.

Each **ABR round** (simultaneous best response):
1. Freeze the opponent's evidence from the previous round.
2. **Player `y`**: greedy hill-climb on `U₂(E_y | E_foil)` — faithfulness minus sparsity minus overlap with the foil set.
3. **Player `y_foil`**: greedy hill-climb on `U₂(E_foil | E_y)` against the **frozen** target evidence from the previous round (Jacobi symmetry: both agents see the same-round opponent, not each other's fresh response).
4. Score the joint allocation `(E_y', E_foil')`, track the best combined utility, and repeat.

Utility for one agent:

`U₂(E) = faithfulness_delta(E) - λ·|E| - β·overlap(E, E_other)`

Under ABR, `overlap` is the hard count `|E ∩ E_other|`.

Below we manually run **ABR round 1** (first inner sweep per agent), score the joint allocation, then preview **round 2** before calling `solve_game2` for the full solver.

In [ ]:
# Game 2 hyperparameters (shared with the full solve below)
GAME2_ALPHA = GAME1_ALPHA
GAME2_LAM = GAME1_LAM
GAME2_BETA = 0.1
GAME2_BUDGET = GAME1_BUDGET
GAME2_ABR_ITERS = 10
GAME2_PREFILTER_TOP_K = GAME1_PREFILTER_TOP_K
GAME2_CONNECTED = GAME1_CONNECTED
GAME2_MIN_GAIN = GAME1_MIN_GAIN
FOIL_ID = "y_foil"

oracle.reset_stats()

# ABR state: round-0 evidence (frozen opponent for round 1)
evidence_y: set = set()
evidence_foil: set = set()

### Step 1: Oracle baselines for target and foil

The same four intervention primitives apply to each target label (`y`, `y_foil`). Game 2 scores faithfulness separately for each agent's evidence set.

In [ ]:
for label in (TARGET_ID, FOIL_ID):
    s_all = oracle.all(label)
    s_empty = oracle.empty(label)
    print(f"[{label}] S_all = {s_all:.6f}, S_empty = {s_empty:.6f}, recoverable = {s_all - s_empty:.6f}")

### Step 2: Game 2 utility at the empty joint allocation

Start with `E_y = E_foil = ∅`. Overlap is zero, so utilities reduce to faithfulness minus sparsity (both zero here).

In [ ]:
from macag.utils.metrics import compute_faithfulness_metrics, game2_utility, overlap_rate
from macag.graph import NodeId


def _sort_key(node: NodeId) -> str:
    return str(node)


def _overlap_weight(nodes: set, other_evidence: set) -> float:
    """ABR hard overlap: count nodes in both sets."""
    return float(len(nodes & other_evidence))


def _evaluate_joint(e_y: set, e_foil: set):
    shared = e_y & e_foil
    m_y = compute_faithfulness_metrics(oracle=oracle, target=TARGET_ID, nodes=e_y, alpha=GAME2_ALPHA)
    m_foil = compute_faithfulness_metrics(oracle=oracle, target=FOIL_ID, nodes=e_foil, alpha=GAME2_ALPHA)
    u_y = game2_utility(
        faithfulness_delta=m_y.faithfulness_delta,
        size=len(e_y),
        overlap_weight=float(len(shared)),
        lam=GAME2_LAM,
        beta=GAME2_BETA,
    )
    u_foil = game2_utility(
        faithfulness_delta=m_foil.faithfulness_delta,
        size=len(e_foil),
        overlap_weight=float(len(shared)),
        lam=GAME2_LAM,
        beta=GAME2_BETA,
    )
    return m_y, m_foil, u_y, u_foil, u_y + u_foil, shared


m_y0, m_foil0, u_y0, u_foil0, combined0, shared0 = _evaluate_joint(evidence_y, evidence_foil)
print(f"E_y = {evidence_y or '∅'},  E_foil = {evidence_foil or '∅'}")
print(f"  shared = {shared0 or '∅'}")
print(f"  U_y = {u_y0:.6f},  U_foil = {u_foil0:.6f},  combined = {combined0:.6f}")

### Step 3: Prefilter with overlap penalty (optional)

Game 2 prefilter ranks singleton utilities using the **full** `U₂` formula, including overlap with the opponent's current evidence. This matches `_prefilter_with_overlap_penalty` in the solver.

For ABR round 1 both opponents are empty, so overlap terms are zero and the ranking matches Game 1 singleton ranking.

In [ ]:
from macag.games.game2_contrastive import _prefilter_with_overlap_penalty

# Round 1: both agents respond to empty opponents
opponent_for_y = {node: 1.0 for node in evidence_foil}
opponent_for_foil = {node: 1.0 for node in evidence_y}

pool_y = _prefilter_with_overlap_penalty(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    fixed_other_weights=opponent_for_y,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    top_k=GAME2_PREFILTER_TOP_K,
    connected=GAME2_CONNECTED,
)
pool_foil = _prefilter_with_overlap_penalty(
    graph=graph,
    oracle=oracle,
    target=FOIL_ID,
    fixed_other_weights=opponent_for_foil,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    top_k=GAME2_PREFILTER_TOP_K,
    connected=GAME2_CONNECTED,
)

print(f"Prefiltered y pool ({len(pool_y)}): {pool_y}")
print(f"Prefiltered foil pool ({len(pool_foil)}): {pool_foil}")

### Step 4: Greedy best response for target agent `y`

Inner loop for player `y`: starting from `E_y = ∅`, try adding each candidate and keep the best **marginal gain** in `U₂(E_y | E_foil^{(0)})`.

When `connected=True`, skip trials that would disconnect the evidence set.

The solver repeats greedy sweeps until no candidate improves `U₂`; here we show the **first sweep** only.

In [ ]:
current_y = set(evidence_y)
current_metrics_y = m_y0
current_u_y = u_y0
best_node_y = None
best_gain_y = GAME2_MIN_GAIN
sweep_rows_y = []

for node in pool_y:
    if node in current_y:
        continue
    trial = set(current_y)
    trial.add(node)
    if GAME2_CONNECTED and len(trial) > 1 and not graph.connected_through(trial):
        continue
    trial_metrics = compute_faithfulness_metrics(
        oracle=oracle, target=TARGET_ID, nodes=trial, alpha=GAME2_ALPHA
    )
    trial_u = game2_utility(
        faithfulness_delta=trial_metrics.faithfulness_delta,
        size=len(trial),
        overlap_weight=_overlap_weight(trial, evidence_foil),
        lam=GAME2_LAM,
        beta=GAME2_BETA,
    )
    gain = trial_u - current_u_y
    sweep_rows_y.append((gain, trial_metrics.faithfulness_delta - current_metrics_y.faithfulness_delta, node))
    if gain > best_gain_y:
        best_gain_y = gain
        best_node_y = node
    elif gain == best_gain_y and best_node_y is not None and _sort_key(node) < _sort_key(best_node_y):
        best_node_y = node

sweep_rows_y.sort(key=lambda r: (-r[0], _sort_key(r[2])))
print(f"[y] sweep evaluated {len(sweep_rows_y)} connected candidates (opponent E_foil = ∅)")
print(f"Current U_y: {current_u_y:.6f}\n")
for gain, faith_gain, node in sweep_rows_y[:5]:
    marker = " <-- winner" if node == best_node_y else ""
    print(f"  {node}: utility_gain={gain:.6f}, faith_gain={faith_gain:.6f}{marker}")

if best_node_y is None:
    print("\nNo improving candidate — target agent would stop with E_y = ∅.")
else:
    print(f"\nBest node for y: {best_node_y}  (utility gain = {best_gain_y:.6f})")

### Step 5: Greedy best response for foil agent `y_foil`

Same inner loop for the foil player, but faithfulness is measured against `y_foil` and overlap is computed against the **frozen** target evidence from round 0 (`E_y = ∅` under Jacobi ABR).

The solver repeats greedy sweeps until no candidate improves `U₂`; here we show the first sweep only.

In [ ]:
current_foil = set(evidence_foil)
current_metrics_foil = m_foil0
current_u_foil = u_foil0
best_node_foil = None
best_gain_foil = GAME2_MIN_GAIN
sweep_rows_foil = []

for node in pool_foil:
    if node in current_foil:
        continue
    trial = set(current_foil)
    trial.add(node)
    if GAME2_CONNECTED and len(trial) > 1 and not graph.connected_through(trial):
        continue
    trial_metrics = compute_faithfulness_metrics(
        oracle=oracle, target=FOIL_ID, nodes=trial, alpha=GAME2_ALPHA
    )
    trial_u = game2_utility(
        faithfulness_delta=trial_metrics.faithfulness_delta,
        size=len(trial),
        overlap_weight=_overlap_weight(trial, evidence_y),
        lam=GAME2_LAM,
        beta=GAME2_BETA,
    )
    gain = trial_u - current_u_foil
    sweep_rows_foil.append((gain, trial_metrics.faithfulness_delta - current_metrics_foil.faithfulness_delta, node))
    if gain > best_gain_foil:
        best_gain_foil = gain
        best_node_foil = node
    elif gain == best_gain_foil and best_node_foil is not None and _sort_key(node) < _sort_key(best_node_foil):
        best_node_foil = node

sweep_rows_foil.sort(key=lambda r: (-r[0], _sort_key(r[2])))
print(f"[y_foil] sweep evaluated {len(sweep_rows_foil)} connected candidates (opponent E_y = ∅)")
print(f"Current U_foil: {current_u_foil:.6f}\n")
for gain, faith_gain, node in sweep_rows_foil[:5]:
    marker = " <-- winner" if node == best_node_foil else ""
    print(f"  {node}: utility_gain={gain:.6f}, faith_gain={faith_gain:.6f}{marker}")

if best_node_foil is None:
    print("\nNo improving candidate — foil agent would stop with E_foil = ∅.")
else:
    print(f"\nBest node for y_foil: {best_node_foil}  (utility gain = {best_gain_foil:.6f})")

### Step 6: Complete inner best responses and score ABR round 1

Steps 4–5 showed the **first greedy sweep** for each agent. The solver's inner `_best_response` subroutine repeats sweeps until no candidate improves `U₂`.

Run the full inner solves (both agents still respond to round-0 opponents under Jacobi ABR), then evaluate the joint allocation and update **best-iterate** tracking.

In [ ]:
from macag.games.game2_contrastive import _best_response

next_y = _best_response(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    fixed_other_weights=opponent_for_y,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    budget=GAME2_BUDGET,
    connected=GAME2_CONNECTED,
    min_gain=GAME2_MIN_GAIN,
    prefilter_top_k=GAME2_PREFILTER_TOP_K,
    progress=True,
    log_every=10,
    progress_desc="ABR[1] y",
)
next_foil = _best_response(
    graph=graph,
    oracle=oracle,
    target=FOIL_ID,
    fixed_other_weights=opponent_for_foil,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    budget=GAME2_BUDGET,
    connected=GAME2_CONNECTED,
    min_gain=GAME2_MIN_GAIN,
    prefilter_top_k=GAME2_PREFILTER_TOP_K,
    progress=True,
    log_every=10,
    progress_desc="ABR[1] foil",
)

m_y1, m_foil1, u_y1, u_foil1, combined1, shared1 = _evaluate_joint(next_y, next_foil)
unique_y = next_y - next_foil
unique_foil = next_foil - next_y

best_evidence_y, best_evidence_foil = set(evidence_y), set(evidence_foil)
best_combined = combined0
best_iteration = 0
if combined1 > best_combined:
    best_combined = combined1
    best_evidence_y, best_evidence_foil = set(next_y), set(next_foil)
    best_iteration = 1

print("ABR round 1 joint allocation:")
print(f"  E_y'    = {next_y or '∅'}")
print(f"  E_foil' = {next_foil or '∅'}")
print(f"  shared      = {shared1 or '∅'}")
print(f"  unique_y    = {unique_y or '∅'}")
print(f"  unique_foil = {unique_foil or '∅'}")
print(f"  overlap_rate = {overlap_rate(next_y, next_foil):.4f}")
print(f"  U_y = {u_y1:.6f}, U_foil = {u_foil1:.6f}, combined = {combined1:.6f}")
print(f"  best_iteration = {best_iteration} (combined = {best_combined:.6f})")

stats = oracle.cache_stats()
print(f"\nOracle calls so far: {stats['oracle_calls']}, cache hits: {stats['cache_hits']}")

### Step 7: ABR round 2 — overlap penalties engage

Advance to round 2: prefilters and best responses now see the **round-1** opponent evidence, so shared nodes incur a `β` penalty before faithfulness is even scored.

If `(E_y^{(2)}, E_{foil}^{(2)})` equals `(E_y^{(1)}, E_{foil}^{(1)})`, the solver sets `converged=True` and stops.

In [ ]:
evidence_y, evidence_foil = set(next_y), set(next_foil)

opponent_for_y_r2 = {node: 1.0 for node in evidence_foil}
opponent_for_foil_r2 = {node: 1.0 for node in evidence_y}

pool_y_r2 = _prefilter_with_overlap_penalty(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    fixed_other_weights=opponent_for_y_r2,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    top_k=GAME2_PREFILTER_TOP_K,
    connected=GAME2_CONNECTED,
)
pool_foil_r2 = _prefilter_with_overlap_penalty(
    graph=graph,
    oracle=oracle,
    target=FOIL_ID,
    fixed_other_weights=opponent_for_foil_r2,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    top_k=GAME2_PREFILTER_TOP_K,
    connected=GAME2_CONNECTED,
)

next_y_r2 = _best_response(
    graph=graph,
    oracle=oracle,
    target=TARGET_ID,
    fixed_other_weights=opponent_for_y_r2,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    budget=GAME2_BUDGET,
    connected=GAME2_CONNECTED,
    min_gain=GAME2_MIN_GAIN,
    prefilter_top_k=GAME2_PREFILTER_TOP_K,
    progress=True,
    log_every=10,
    progress_desc="ABR[2] y",
)
next_foil_r2 = _best_response(
    graph=graph,
    oracle=oracle,
    target=FOIL_ID,
    fixed_other_weights=opponent_for_foil_r2,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    budget=GAME2_BUDGET,
    connected=GAME2_CONNECTED,
    min_gain=GAME2_MIN_GAIN,
    prefilter_top_k=GAME2_PREFILTER_TOP_K,
    progress=True,
    log_every=10,
    progress_desc="ABR[2] foil",
)

_, _, _, _, combined2, shared2 = _evaluate_joint(next_y_r2, next_foil_r2)
if combined2 > best_combined:
    best_combined = combined2
    best_evidence_y, best_evidence_foil = set(next_y_r2), set(next_foil_r2)
    best_iteration = 2

converged = next_y_r2 == evidence_y and next_foil_r2 == evidence_foil
print(f"ABR round 2: converged={converged}")
print(f"  E_y''    = {next_y_r2 or '∅'}")
print(f"  E_foil'' = {next_foil_r2 or '∅'}")
print(f"  shared = {shared2 or '∅'}, combined utility = {combined2:.6f}")
print(f"  best_iteration = {best_iteration}")

## Run full Game 2

Now run the complete ABR solver (all rounds, caching, best-iterate tracking, and optional fictitious-play mode via `solver="fp"`).

In [ ]:
from macag.games.game2_contrastive import solve_game2

result_game2 = solve_game2(
    graph=graph,
    oracle=oracle,
    y=TARGET_ID,
    y_foil=FOIL_ID,
    candidates=candidates,
    alpha=GAME2_ALPHA,
    lam=GAME2_LAM,
    beta=GAME2_BETA,
    abr_iters=GAME2_ABR_ITERS,
    budget=GAME2_BUDGET,
    connected=GAME2_CONNECTED,
    min_gain=GAME2_MIN_GAIN,
    prefilter_top_k=GAME2_PREFILTER_TOP_K,
    solver="abr",
    progress=True,
    log_every=10,
)

print(f"|E_y| = {len(result_game2.evidence_y)}, |E_foil| = {len(result_game2.evidence_foil)}")
print(f"shared      = {result_game2.shared or '∅'}")
print(f"unique_y    = {result_game2.unique_y or '∅'}")
print(f"unique_foil = {result_game2.unique_foil or '∅'}")
print(f"overlap_rate = {result_game2.overlap_rate:.4f}")
print(f"U_y = {result_game2.utility_y:.6f}, U_foil = {result_game2.utility_foil:.6f}")
print(f"combined utility = {result_game2.utility_y + result_game2.utility_foil:.6f}")
print(f"faithfulness_y = {result_game2.metrics_y.faithfulness_delta:.6f}")
print(f"faithfulness_foil = {result_game2.metrics_foil.faithfulness_delta:.6f}")
print(f"best_iteration = {result_game2.best_iteration} (0 = empty allocation won)")
print(f"converged = {result_game2.converged}, iterations = {result_game2.iterations}")
print(f"Oracle calls: {result_game2.oracle_calls}, cache hits: {result_game2.cache_hits}")

## Visualize Game 2 evidence on the graph

Annotate the graph with `shared`, `unique_y`, and `unique_foil` supernodes, then open it in the circuit-tracer frontend (same pattern as Game 1 above).

In [ ]:
# from IPython.display import IFrame, display
# from circuit_tracer.frontend.local_server import serve
#
# annotated_path = Path(OUTDIR) / "graphs" / f"macag-{SLUG}"
# # Write macag_game2.json from result_game2 via CLI serializer, then:
# # python -m macag.cli.annotate_graph --graph-json ... --macag-result-json ...
# # (omit --output-json to auto-write graphs/macag-<slug>.json with a MACAG title prefix)
#
# PORT = 8042
# server = serve(data_dir=str(graph_dir.resolve()), port=PORT)
# url = f"http://localhost:{PORT}/index.html?slug={annotated_path.name}"
# print(f"Open in browser: {url}")
# display(IFrame(src=url, width="100%", height="800px"))